# install deps
- opencv-python: to read and process images and videos
- matplotlib: to visualize images and videos, render the results so that we can see outputs of our preprocessed videos
- imageio: to create a gif from the processed video frames to see a couple of frames stacked together
- gdown: to download the dataset from google drive
- tensorflow: to build and train the DNN

In [ ]:
%pip install opencv-python matplotlib imageio gdown tensorflow

In [ ]:
!pip list

In [ ]:
import os
import cv2 # to read and process videos
import tensorflow as tf # primary DL framework, for data pipeline and model building
import numpy as np # to preprocess any arrays
from typing import List # type annotation
import imageio # convert numpy array to gif

# 1. build data loading function

We are going to use GRID dataset for our lip reading model. This dataset is something that can later be used with a custom dataset, say videos of ourselves. We just need to capture frames of a person speaking, then use a speech to text model to transcribe what the person is saying. That dataset can then be subbed into the model training pipeline. We need two data loading functions, one to load the videos and another to load the aligned transcriptions.

In [ ]:
import gdown

In [ ]:
# url = 'https://drive.google.com/uc?id=1YlvpDLix3S-U8fd-gqRwPcWXAXm8JwjL'
# output = 'data.zip'
# gdown.download(url, output, quiet=False)
# gdown.extractall('data.zip')

Now we have something here called alignments in the dataset and there is .align file for each video, which is like annotations for their corresponding video. The .align file contains the start and end times of each word spoken in the video, along with the actual word. This is crucial for training our lip reading model, as it allows us to know exactly which frames correspond to which words.

Here sil is silence, bin is the word "bin", blue is the word "blue", at is the word "at", f is the word "f", two is the word "two", and now is the word "now". The numbers represent the start and end times of each word in milliseconds.

We got to train a model that is able to decode this purely from a vid and no audio. We'll remove the sil part and we want to purely extract the middle part 
```
0 23750 sil
23750 29500 bin
29500 34000 blue
34000 35500 at
35500 41000 f
41000 47250 two
47250 53000 now
53000 74500 sil
```

In [ ]:
def load_video(path:str) -> List[float]:

    cap = cv2.VideoCapture(path) # reads the video frame by frame
    frames = [] # empty py list, each processed frame will be put here as a numpy array
    for _ in range(int(cap.get(cv2.CAP_PROP_FRAME_COUNT))): # the loop runs for the number of frames in the video
        ret, frame = cap.read() # ret is a boolean indicating if the frame was read successfully, and frame is the actual frame
        frame = tf.image.rgb_to_grayscale(frame) # converts from rgb to grayscale to reduce the number of channels from 3 to 1, so less preprocessing
        frames.append(frame[190:236,80:220,:]) # crops the frame to focus on the region of interest, which is the hand in this case, hardcoding possible because the camera is fixed and the person is always in the same position
    cap.release()

    # Normalization part
    mean = tf.math.reduce_mean(frames) # avg pixel value accross the frames
    std = tf.math.reduce_std(tf.cast(frames, tf.float32)) # how much the pixel values vary from avg
    return tf.cast((frames - mean), tf.float32) / std # int(pixel data) to float32 conversion,


## Standard deviation

How much the pixel values vary around that average. Then: `(frames - mean) / std` is standardization.
Mathematically:

$$ x' = \frac{x-\mu}{\sigma} $$

This roughly makes the data have:
mean ≈ 0
standard deviation ≈ 1

Why? Because neural networks generally train more effectively when input values are normalized rather than having arbitrary raw pixel ranges.

### tf.cast part
The code converts them to `float32` because neural-network calculations are generally performed using floating-point values.

So: `tf.cast(frames, tf.float32)` means Convert the pixel data to 32-bit floating point.

## 18. Now Why List[float]?

This part is actually a little misleading in the tutorial.
`def load_video(path:str) -> List[float]:` means, according to Python type hints: "This function returns a list containing floats." But that's not really what this function returns.
Inside the function: `frames = []` and then: `frames.append(frame[...])`

So frames contains images/tensors, not individual floats.

And finally: `return tf.cast((frames - mean), tf.float32) / std` returns a TensorFlow tensor, not a Python List[float]. So this:  `List[float]` is essentially an inaccurate type annotation in this tutorial.
Don't take it as: "The video is converted into a list of floats." The actual conceptual output is:

```
Video
 ↓
many frames
 ↓
each frame = numerical pixel tensor
 ↓
all frames = 4D tensor approximately:
(number_of_frames, height, width, channels)
```

For example, after cropping:

```
90 frames
×
46 height
×
140 width
×
1 channel
```

So approximately: (90, 46, 140, 1), That's what is eventually fed into the neural network.

vocab range that we'll encounter 

In [ ]:
vocab = [x for x in "abcdefghijklmnopqrstuvwxyz'?!123456789 "]

In [ ]:
char_to_num(['n', 'i', 'c', 'k'])

In [ ]:
num_to_char([14,  9,  3, 11])

In [ ]:
char_to_num.get_vocabulary()

## StringLookup from Keras CTC ASR (automatic speech recognition)

can use this to look up or convert num to char or char to num

In the above, using char_to_num, we get the following output and also using num_to_char, we get the intended output and you can see that it's tokenizing it and returning a specific token value or index effectively.  we are going to be able to pass through this data to our loss function so as to calculate our overall loss because our model will be returning a one hot encoded version of this

To decode the output, we'll use the num_to_char function to convert the output back to characters. This is important because our model will output a sequence of numbers, and we need to convert them back to characters to understand what the model is predicting.

```python
char_to_num(['n', 'i', 'c', 'k'])
# output: <tf.Tensor: shape=(4,), dtype=int64, numpy=array([14,  9,  3, 11])>

num_to_char([14,  9,  3, 11])
# output: <tf.Tensor: shape=(4,), dtype=string, numpy=array([b'n', b'i', b'c', b'k'], dtype=object)>
# byte encoded value
```

The model needs numerical data.  Think of it like this:

```
Human-readable:   "nick"
          ↓
Machine-readable: [14, 9, 3, 11]
```

And after prediction:

```
Model output: [14, 9, 3, 11]
          ↓
Human-readable: "nick"
```

So char_to_num and num_to_char are basically the translator between text and numbers.

In [ ]:
char_to_num = tf.keras.layers.StringLookup(vocabulary=vocab, oov_token="")
num_to_char = tf.keras.layers.StringLookup(
    vocabulary=char_to_num.get_vocabulary(), oov_token="", invert=True # layer maps from integer indices to string tokens
)

print(
    f"The vocabulary is: {char_to_num.get_vocabulary()} "
    f"(size ={char_to_num.vocabulary_size()})"
)

In [ ]:
def load_alignments(path: str) -> List[str]:
    with open(path, 'r') as f:
        lines = f.readlines() # read line by line
    tokens = []
    # We are going to read and split out each one of those lines into individual tokens, and then we will return a list of those tokens.
    for line in lines: # Go through the alignment file one line at a time.
        line = line.split() # split the line into individual tokens
        if line[2] != 'sil': # remove the silence 'sil' part
            tokens = [*tokens, ' ', line[2]] # this is just for representing like a sentence
            # the ' ' is added so that the words remain separate

    return char_to_num(tf.reshape(tf.strings.unicode_split(tokens, 'UTF-8'), (-1)))[1:] # convert the tokens to integers using the char_to_num layer
    # tf.reshape(..., (-1)) basically flattens the character sequence into one dimension.
    # [1: ] is to remove the first element which is a space, because we don't want to start with a space in our sequence.

```
ALIGNMENT FILE
      ↓
spoken words
      ↓
"set red"
      ↓
characters
      ↓
s e t   r e d
      ↓
character IDs
      ↓
[19,5,20,...]
ALIGNMENT FILE
      ↓
spoken words
      ↓
"set red"
      ↓
characters
      ↓
s e t   r e d
      ↓
character IDs
      ↓
[19,5,20,...]
```

In [ ]:
def load_data(path: str):
    path = bytes.decode(path.numpy())
    # for unix file path style
    file_name = path.split('/')[-1].split('.')[0]
    # for windows file path style
    # file_name = path.split('\\')[-1].split('.')[0]
    video_path = os.path.join('data','s1',f'{file_name}.mpg')
    alignment_path = os.path.join('data','alignments','s1',f'{file_name}.align')
    frames = load_video(video_path)
    alignments = load_alignments(alignment_path)

    return frames, alignments

Find the video
`video_path = os.path.join( 'data', 's1', f'{file_name}.mpg')`
Result: data/s1/bbal6n.mpg

Find its alignment file
`alignment_path = os.path.join( 'data', 'alignments', 's1', f'{file_name}.align')`
Result: data/alignments/s1/bbal6n.align

This is important: The video and alignment file have the same filename.

bbal6n.mpg
bbal6n.align

So the program knows which text corresponds to which video.

In [ ]:
# load_data(tf.constant('data/s1/bbaf2n.mpg'))

In [ ]:
test_path = '../data/s1/bbal6n.mpg'

In [ ]:
tf.convert_to_tensor(test_path).numpy().decode('utf-8').split('/')[-1].split('.')[0]

In [ ]:
frames, alignments = load_data(tf.convert_to_tensor(test_path))

In [ ]:
plt.imshow(frames[37])

In the above image we can see the lips moving and this is the impact of the sub the mean and the standard deviation. We are isolating the lips and the mouth region and this is the part that is going to be used for training the model. The model will learn to recognize the movements of the lips and mouth and associate them with the corresponding words in the alignment file.

The alignments processing gives the numerical representation of the spoken sentence.
```
<tf.Tensor: shape=(21,), dtype=int64, numpy=
array([ 2,  9, 14, 39,  2, 12, 21,  5, 39,  1, 20, 39, 12, 39, 19,  9, 24,
       39, 14, 15, 23])>
```

In [ ]:
alignments

In [ ]:
tf.strings.reduce_join(num_to_char(alignments.numpy()))
# print([bytes.decode(x) for x in num_to_char(alignments.numpy()).numpy()])

### mappable_function()

Why is this needed? Because notebook uses: `tf.data.Dataset` and TensorFlow's dataset pipeline expects TensorFlow-compatible functions.  But load_data() contains normal Python operations: `open()`, `path.numpy()`, and `cv2.VideoCapture()`. TensorFlow cannot directly trace all of those operations as a normal TensorFlow graph.

So: `tf.py_function(...)` basically tells TensorFlow: "Run this function as normal Python, then give me the result back as TensorFlow tensors."

What are these arguments?
tf.py_function(
    load_data,
    [path],
    (tf.float32, tf.int64)
)

They mean- Function: load_data Input: [path]
Expected outputs: (tf.float32, tf.int64) Because:

```
load_data()
    ↓
frames       → float32
alignments   → int64
```

In [ ]:
tf.strings.reduce_join([bytes.decode(x) for x in num_to_char(alignments.numpy()).numpy()])

def mappable_function(path:str) ->List[str]:
    result = tf.py_function(load_data, [path], (tf.float32, tf.int64)) # returns float64 and int64 tensors, and also allows raw string processing on the tensor, which is not possible with tf.function. Pure string processing
    return result

# 2. Data Pipeline

We need to build a data pipeline. This will be used to train the deep learning model. TensorFlow will draw random samples from our dataset in order to complete one training step. we also need to look at the data to make sure our transformations have worked successfully. 

In [ ]:
from matplotlib import pyplot as plt # to render our post-process output of our data loading function

following is one of the important bits of the entire thing here.

- list_files() lists all the video files in the data/s1 directory.  
- shuffle(500) is the cache size, which is the number of elements from this dataset from which the new dataset will sample. The larger the cache size, the more randomness in the sampling, but also more memory usage.
- data.map(mappable_function) applies the mappable_function to each element in the dataset, check the mappable_function() above.
- data.padded_batch(2, padded_shapes=([75,None,None,None],[40])) groups the dataset into batches of 2, so 2 vids and 2 aligns. Since the dataset contains variable length data, so the batches will be padded to ensure all sequences have the same length. Also there must be 40 tokens for each one of our alignments, and if not it'll be padded to 0. None for the other three ensures we don't pad the height, width, and channels of the video frames. The first dimension (75) is the maximum number of frames in a video, and the second dimension (40) is the maximum number of tokens in an alignment. If a video has fewer than 75 frames or an alignment has fewer than 40 tokens, they will be padded with zeros to reach these lengths.

In [ ]:
data = tf.data.Dataset.list_files('./data/s1/*.mpg') # list all the video files in the data/s1 directory
data = data.shuffle(500)
data = data.map(mappable_function)
data.padded_batch(2, padded_shapes=([75,None,None,None],[40]))

In [ ]:
frames, alignments = data.as_numpy_iterator().next()